# Mô Phỏng Máy Hút Bụi (Vacuum Cleaner Agent) – Random Restart Hill Climbing

**Quy ước ma trận:**
- `0` → Ô trống / Máy hút bụi
- `1` → Bụi
- `2` → Tường / Vật cản

**Chiến lược Random Restart Hill Climbing:**
- Nếu ô hiện tại có bụi → hút ngay.
- Dùng Random Restart Hill Climbing để tìm đường đi ngắn nhất đến ô bụi gần nhất:
  - Hàm lượng giá (Heuristic): $h(n)$ = khoảng cách Manhattan từ ô $n$ đến ô bụi đích.
  - Chọn láng giềng có $h(n)$ tốt nhất (Steepest Ascent).
  - Nếu bị kẹt tại cực đại cục bộ (không có láng giềng nào có $h(n)$ tốt hơn ô hiện tại), thực hiện **Random Restart**:
    - Bắt đầu lại từ vị trí ban đầu (start) nhưng đi ngẫu nhiên một số bước (random walk) để thoát khỏi khu vực bị kẹt, sau đó tiếp tục leo núi.

In [ ]:
import numpy as np
import random


In [ ]:
# ── Cấu hình ──
ROWS      = 5
COLS      = 7
WALL_PROB = 0.15
DUST_PROB = 0.35
MAX_STEPS = 300

# ── Tạo môi trường ngẫu nhiên ──
def create_env(rows, cols, wall_prob, dust_prob):
    grid = np.zeros((rows, cols), dtype=int)
    for r in range(rows):
        for c in range(cols):
            v = random.random()
            if v < wall_prob:
                grid[r][c] = 2
            elif v < wall_prob + dust_prob:
                grid[r][c] = 1
    free = [(r, c) for r in range(rows) for c in range(cols) if grid[r][c] != 2]
    pos = random.choice(free)
    return grid, pos

grid, start = create_env(ROWS, COLS, WALL_PROB, DUST_PROB)
total_dust = int(np.sum(grid == 1))

print(f"Ma trận ban đầu (vị trí máy: {start}):")
print(grid)
print(f"Tổng bụi: {total_dust} ô")


In [ ]:
# ── Agent (Random Restart Hill Climbing) ──
MOVES = {'UP': (-1,0), 'DOWN': (1,0), 'LEFT': (0,-1), 'RIGHT': (0,1)}

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def hill_climbing_find_nearest_dust(grid, start_pos):
    """
    Tìm đường đi đến ô bụi gần nhất bằng thuật toán Random Restart Hill Climbing.
    """
    start_pos = tuple(start_pos)
    rows, cols = grid.shape
    dust_cells = [(r, c) for r in range(rows) for c in range(cols) if grid[r, c] == 1]
    if not dust_cells:
        return None, None

    # Tìm ô bụi gần nhất làm đích
    best_target = None
    min_dist = float('inf')
    for dust in dust_cells:
        d = manhattan(start_pos, dust)
        if d < min_dist:
            min_dist = d
            best_target = dust

    if best_target is None:
        return None, None

    max_restarts = 100
    for attempt in range(max_restarts):
        path = [start_pos]
        visited = {start_pos}

        # Nếu là restart (attempt > 0), thực hiện một số bước ngẫu nhiên (random walk) để thoát kẹt
        if attempt > 0:
            curr = start_pos
            random_steps = random.randint(1, 5)
            for _ in range(random_steps):
                neighbors = []
                for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nr, nc = curr[0] + dr, curr[1] + dc
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] != 2 and (nr, nc) not in visited:
                        neighbors.append((nr, nc))
                if not neighbors:
                    break
                next_node = random.choice(neighbors)
                path.append(next_node)
                visited.add(next_node)
                curr = next_node

        curr = path[-1]
        stuck = False
        
        while curr != best_target:
            neighbors = []
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = curr[0] + dr, curr[1] + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] != 2 and (nr, nc) not in visited:
                    neighbors.append((nr, nc))

            if not neighbors:
                stuck = True
                break

            # Lượng giá các láng giềng
            best_n = []
            best_val = float('inf')
            for n in neighbors:
                val = manhattan(n, best_target)
                if val < best_val:
                    best_val = val
                    best_n = [n]
                elif val == best_val:
                    best_n.append(n)

            # So sánh với ô hiện tại
            curr_val = manhattan(curr, best_target)
            if best_val < curr_val:
                next_node = random.choice(best_n)
                path.append(next_node)
                visited.add(next_node)
                curr = next_node
            else:
                stuck = True
                break

        if not stuck and curr == best_target:
            move_path = []
            for i in range(len(path) - 1):
                r1, c1 = path[i]
                r2, c2 = path[i+1]
                dr, dc = r2 - r1, c2 - c1
                for d, (mdr, mdc) in MOVES.items():
                    if mdr == dr and mdc == dc:
                        move_path.append((d, dr, dc))
                        break
            return move_path, best_target

    # Thử các ô bụi khác nếu ô bụi gần nhất không thể tiếp cận được
    for target in dust_cells:
        if target == best_target:
            continue
        for attempt in range(20):
            path = [start_pos]
            visited = {start_pos}

            if attempt > 0:
                curr = start_pos
                random_steps = random.randint(1, 4)
                for _ in range(random_steps):
                    neighbors = []
                    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                        nr, nc = curr[0] + dr, curr[1] + dc
                        if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] != 2 and (nr, nc) not in visited:
                            neighbors.append((nr, nc))
                    if not neighbors:
                        break
                    next_node = random.choice(neighbors)
                    path.append(next_node)
                    visited.add(next_node)
                    curr = next_node

            curr = path[-1]
            stuck = False
            while curr != target:
                neighbors = []
                for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nr, nc = curr[0] + dr, curr[1] + dc
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] != 2 and (nr, nc) not in visited:
                        neighbors.append((nr, nc))

                if not neighbors:
                    stuck = True
                    break

                best_n = []
                best_val = float('inf')
                for n in neighbors:
                    val = manhattan(n, target)
                    if val < best_val:
                        best_val = val
                        best_n = [n]
                    elif val == best_val:
                        best_n.append(n)

                curr_val = manhattan(curr, target)
                if best_val < curr_val:
                    next_node = random.choice(best_n)
                    path.append(next_node)
                    visited.add(next_node)
                    curr = next_node
                else:
                    stuck = True
                    break

            if not stuck and curr == target:
                move_path = []
                for i in range(len(path) - 1):
                    r1, c1 = path[i]
                    r2, c2 = path[i+1]
                    dr, dc = r2 - r1, c2 - c1
                    for d, (mdr, mdc) in MOVES.items():
                        if mdr == dr and mdc == dc:
                            move_path.append((d, dr, dc))
                            break
                return move_path, target

    return None, None

def run_agent(grid_in, start, max_steps):
    grid = grid_in.copy()
    rows, cols = grid.shape
    pos = list(start)
    history = set()
    steps = 0
    cleaned = 0

    while steps < max_steps:
        r, c = pos

        # Hút bụi tại ô hiện tại nếu có
        if grid[r][c] == 1:
            state = ((r, c), 'CLEAN')
            if state in history:
                return grid, steps, cleaned, 'THAT BAI', 'Lap lai hanh dong CLEAN tai ' + str((r, c))
            history.add(state)
            grid[r][c] = 0
            cleaned += 1
            steps += 1
            if cleaned == total_dust:
                return grid, steps, cleaned, 'THANH CONG', 'Da hut sach toan bo bui'
            continue

        # Tìm đường đi đến ô bụi gần nhất bằng Random Restart Hill Climbing
        path, target = hill_climbing_find_nearest_dust(grid, pos)
        if path is None or len(path) == 0:
            if cleaned == total_dust:
                return grid, steps, cleaned, 'THANH CONG', 'Da hut sach toan bo bui'
            return grid, steps, cleaned, 'THAT BAI', 'Khong tim thay duong di den bui'

        # Thực hiện bước đi đầu tiên trong đường đi tìm được
        direction, dr, dc = path[0]
        state = ((r, c), direction)
        if state in history:
            return grid, steps, cleaned, 'THAT BAI', f'Lap lai hanh dong {direction} tai {(r, c)}'
        history.add(state)

        pos = [r + dr, c + dc]
        steps += 1

    return grid, steps, cleaned, 'THAT BAI', f'Vuot qua gioi han {max_steps} buoc'


In [ ]:
final_grid, steps, cleaned, status, reason = run_agent(grid, start, MAX_STEPS)

print("Ma tran sau khi chay:")
print(final_grid)
print()
print(f"So buoc di : {steps}")
print(f"Bui da hut : {cleaned} / {total_dust} o")
print(f"Trang thai : {status}")
print(f"Ly do      : {reason}")
